# CS 6320 Project 4: Depth Estimation using Stereo

Two cameras looking at the same scene from slightly different positions see each object at a
slightly different horizontal position. That shift is the **disparity**, and it is inversely
proportional to depth, which is how a stereo rig, or a pair of eyes, recovers 3D structure from
two flat images.

You can see the effect without any equipment. Look at a scene with only your left eye, then switch
quickly to only your right. Objects jump sideways, and **near objects jump further than far ones**.
That difference in jump is exactly what you are about to compute at every pixel.

In this project you will build a block-matching stereo algorithm from scratch, study where it
succeeds and where it fails, and then improve it with a smoothing constraint.

**This notebook is your report.** Run every cell before submitting, anything not visible in the
saved notebook cannot be graded.

## Setup

1. Install [Miniconda](https://docs.conda.io/en/latest/miniconda.html) if you do not already have
   conda.
2. Create the environment, substituting your OS (`linux`, `mac`, or `win`):
   `conda env create -f proj4_env_<OS>.yml`
3. Activate it: `conda activate proj4`
4. Install the project package from the repo folder: `pip install -e .`
5. Start Jupyter and open the notebook: `jupyter notebook ./proj4_code/simple_stereo.ipynb`
6. Check your work as you go with the `verify(...)` calls built into the notebook. To run the whole
   suite at once, from the **repo root**: `pytest unit_tests`
7. When everything is done, run every cell top to bottom, save, and build your submission:
   `python zip_submission.py --uid <your_uid>`, then upload the zip to Gradescope.



## What you hand in

Answer every reflection question in the markdown cell directly beneath it. A good answer explains
*why* the result looks the way it does, in terms of the algorithm and the image content, not just
what you see. A few sentences is usually enough.

`zip_submission.py` packages `proj4_code/`, `unit_tests/`, `semiglobalmatching/`, and `data/`, so
your code and your run notebook all go up together. Use relative paths only.

## Rubric

**Code (60 points)**

| Item | Points |
| --- | --- |
| `generate_random_stereogram()` in `utils.py` | 15 |
| `ssd_similarity_measure()` and `sad_similarity_measure()` in `similarity_measures.py` | 15 |
| `calculate_disparity_map()` in `disparity_map.py` | 20 |
| `calculate_cost_volume()` in `disparity_map.py` | 10 |

**Writeup (40 points)**, across the sixteen reflection questions:

| Question | Pts | | Question | Pts |
| --- | --- | --- | --- | --- |
| Q1 disparity and depth | 3 | | Q9 bowling pin head | 2 |
| Q2 judging depth from a stereogram | 2 | | Q10 zero disparity in Set 4 | 3 |
| Q3 effect of block size | 3 | | Q11 the left-border band | 2 |
| Q4 the border of the square | 3 | | Q12 block size and the stairs | 2 |
| Q5 convex error profiles | 3 | | Q13 SGM versus no smoothing | 3 |
| Q6 non-convex error profiles | 3 | | Q14 where smoothing helps | 2 |
| Q7 best patch size | 2 | | Q15 where smoothing hurts | 2 |
| Q8 the chair backrest | 3 | | Q16 vertical shift | 2 |

**Part 4, extra credit (+10 points).** Score how much each pixel's disparity can be trusted,
using the cost volume you already built.

| Item | Points |
| --- | --- |
| `confidence_from_cost_volume()` in `disparity_map.py`, both methods | 4 |
| Q17 reading the confidence maps | 3 |
| Q18 the keep-rate curve and a recommendation | 3 |

**Part 5, required, 0 points.** Complete the AI-use disclosure at the end of the notebook,
including if you used no AI tools.

**Deductions.** −5 points each time the submission instructions are not followed (wrong file name,
missing outputs, absolute paths, missing AI-use disclosure).

# Part 1: Simple stereo by matching patches

In [ ]:
import sys
sys.path.append('..')
from unit_tests.test_base import verify

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from proj4_code.utils import (load_image, PIL_resize, generate_random_stereogram,
                              stereo_helper_fn)
from proj4_code.disparity_map import (calculate_disparity_map, calculate_cost_volume,
                                      confidence_from_cost_volume)
from proj4_code.similarity_measures import ssd_similarity_measure, sad_similarity_measure


%load_ext autoreload
%autoreload 2

In [ ]:
# Supressing warnings about memory leaks is flatly a bad idea and I won't do it.
# See https://piazza.com/class/msy68j54poz4vq/post/66
## # Keep figures from piling up across the many plots in this notebook.
## plt.rcParams['figure.max_open_warning'] = 0

`stereo_helper_fn` from `utils.py` computes and plots disparity maps for a list
of block sizes, using **both** similarity measures, so you can compare SAD and SSD side by side at
each setting.

## Random dot stereogram

It was once believed that perceiving depth required first recognizing *objects*, matching
distinctive features between the two eyes, or reading cues like shading and perspective.

The **random dot stereogram** experiment disproved this. Both images are pure noise:
there are no objects, no edges, no shading, nothing a feature detector could latch onto. Yet a
human viewing them stereoscopically sees a shape floating in depth. Stereo correspondence alone is
enough.

A random dot stereogram is built like this:

1. Fill the left image with random dots (values 0 or 1).
2. Copy it to make the right image.
3. Shift a rectangular region of the right image horizontally by the disparity.
4. Fill the gap left behind with fresh random values.

Implement this as `generate_random_stereogram()` in `utils.py`. The unit test is in
`test_utils.py`. Read the docstring carefully, the block position, the direction of the shift, and
the requirement that all channels be identical are all checked.

In [ ]:
from IPython.core.magic import register_cell_magic
from IPython import get_ipython
from importlib.util import find_spec

@register_cell_magic
def needs(line, cell):
    """Cells with `%%needs a b c` run if a-c _all_ exist and are truthy."""
    if all(globals().get(key) for key in line.split()):
        get_ipython().run_cell(cell)

@register_cell_magic
def depends(line, cell):
    """Cells with `%%depends a b c` run if a-c are _all_ importable."""
    if all(find_spec(key) for key in line.split()):
        get_ipython().run_cell(cell)

In [ ]:
plt.style.use('dark_background')

In [ ]:
%%depends plotly
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.templates.default = 'plotly_dark'

### Reflection Question 1 (3 pts)
What is the relation between the disparity map and depth? Give the equation and describe the relation.

**Answer:** From equation 12.1: $$d=f\frac{B}{Z} \implies Z=f\frac{B}{d}$$. Simply, disparity is inversely proportional to depth. 0 disparity sets the "background" plane at infinity and a low disparity (of a few pixels) brings the rectange forward from infinity.

In [ ]:
from unit_tests.test_utils import test_generate_random_stereogram

print('Test for random dot stereogram:', verify(test_generate_random_stereogram))


In [ ]:
im_left, im_right = generate_random_stereogram(im_size=(256, 256, 3), disparity=16)
print('left:', tuple(im_left.shape), ' right:', tuple(im_right.shape))
print('unique values:', torch.unique(im_left).tolist())


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
ax1.imshow(im_left * 255)
ax1.set_title('Left image')
ax1.axis('off')
ax2.imshow(im_right * 255)
ax2.set_title('Right image')
ax2.axis('off')
plt.tight_layout()
plt.show()

### Reflection Question 2 (2 pts)
Look at the two 51×51×3 images you just generated. Can you tell there is a shape in there by eye?
Why or why not, and what does that tell you about what the matching algorithm is doing that your
visual system, looking at one image at a time, cannot?

**Answer:** Yes, I can see the image, although increasing the image resolution helped quite a bit. The "matching algorithm" in my binocular vision uses disparity to perceive depth, which requires each eye to have a slightly different view of the same scene.

## Similarity measures

To match a patch in the left image against candidates in the right, we need a way to score how
different two patches are. You will implement two:

1. **Sum of Squared Differences:** $\mathrm{SSD}(A, B) = \sum_{i,j}(A_{ij} - B_{ij})^2$
2. **Sum of Absolute Differences:** $\mathrm{SAD}(A, B) = \sum_{i,j}|A_{ij} - B_{ij}|$

Both return a single scalar: **lower means more similar**. They differ in how harshly they punish
a large mismatch. SSD squares the difference, so one badly mismatched pixel can dominate the score;
SAD grows only linearly, which makes it more tolerant of outliers such as a specular highlight or
an occluded pixel. You will see this difference in the disparity maps later.

Implement them in `similarity_measures.py`. The unit tests are in `test_similarity_measures.py`.

### Run the similarity measure tests

In [ ]:
from unit_tests.test_similarity_measures import (
    test_ssd_similarity_measure_values,
    test_sad_similarity_measure_values,
    test_similarity_measure_size_compatibility
)

print('SAD values:          ', verify(test_sad_similarity_measure_values))
print('SSD values:          ', verify(test_ssd_similarity_measure_values))
print('Input size handling: ', verify(test_similarity_measure_size_compatibility))


## Disparity map

Now the core algorithm.

<figure>
    <img src="figures/disparity_calculation_example.jpg" width="600" title="Example of a stereo algorithm" />
    <center><figcaption align="center">Example of a stereo algorithm</figcaption></center>
</figure>

The steps shown in that figure, which you will implement:

1. Take a patch from the left image (red block), centered on the pixel whose disparity you want.
2. Place that patch at the **same** (row, column) in the right image (red block).
3. Slide the candidate window **leftward** from there (the pink blocks), one pixel at a time, up to
   `max_search_bound` pixels. Candidates overlap.
4. Score each candidate with your similarity function and keep the best one (green block). The
   horizontal distance from the starting position to the winner is the disparity at that pixel.

**Search leftward.** In a rectified stereo pair with the cameras side by side, a
point in the world always appears *further right* in the left image than in the right one. So the
match for a left-image patch is always at the same column or to the left of it, searching only one direction halves the work. 

**Ties.** If several disparities give exactly the same minimum error, return the **smallest**
disparity. Check how `argmin` behaves in your chosen library before relying on it.

The images are already **rectified**, which is what lets us search along a horizontal line rather
than a general epipolar line.

Implement `calculate_disparity_map()` in `disparity_map.py`; the unit tests are in
`test_disparity_map.py`.

In [ ]:
from unit_tests.test_disparity_map import (
    generate_delta_fn_images,
    test_disparity_deltafn_failure,
    test_disparity_deltafn_success,
    test_disparity_map_size,
    test_disparity_random_stereogram,
    test_disparity_translation_shift
)

print('Delta function, search bound too small:', verify(test_disparity_deltafn_failure))
print('Delta function, search bound adequate:', verify(test_disparity_deltafn_success))
print('Output size:                          ', verify(test_disparity_map_size))
print('Random dot stereogram:                ', verify(test_disparity_random_stereogram))
print('Pure translation:                     ', verify(test_disparity_translation_shift))


In [ ]:
!python -m this | grep -i namespaces
from proj4_code import utils
from proj4_code import disparity_map as dm

In [ ]:
stereo_helper_fn(im_left*255, im_right*255, block_size=[3, 5, 9, 13], max_search_bound=32)

### Reflection Question 3 (3 pts)
What is the effect of increasing the block size? Explain the mechanism, not just the appearance.

**Answer:** A larger block size makes the similarity function less sensitive to noise and small features. With our salt-and-pepper noise, a block of size b has a $2^{-(b^2)}$ chance of a false match. For a 3x3 block this is a significant amount; with a 5x5 block the probability is negligible.

Conversely, a larger block size struggles to capture fine features, and overestimates the left edge of the moved block by almost the entire block width. Tjis is explained by the fact that a large block is offset by $b-1$ pixels, a single overlapping column of $b$ pixels may be enough to make a correspondance, padding the matched region by $b-1$

### Reflection Question 4 (3 pts)
Look at the random dot stereogram disparity maps. The interior of the shifted square and the
background around it both come out right, but there is a ragged border of wrong values where the
two meet — clearest at block size 3 and 5. Give **two** separate reasons for it. For the first,
think about what a patch sitting on the boundary contains. For the second, think about which parts
of the scene appear in both images and which appear in only one.

**Answer:**
1. The border is "ragged" when the block covers two regions - one that has shifted and one that has not. The similarity metric may choose either value, or some nonsensical value, depending on how well the randomness of either region aligns with the other's search target.
2. The left edge of the box in the disparity map shows pixels that are only visible in the left image. These pixels have no correspondence in the right image, and so get randomly distributed disparities. This could be resolved by tracking the disparity _values_ alongside _indices_ (`torch.min()` computes both together anyway), and discarding disparities with low similarity.

## Error profile analysis

Before computing a whole disparity map, it is worth looking at what the matcher actually sees for
a **single** patch: the similarity error as a function of horizontal shift. That curve is the
*error profile*, and its shape determines whether the match will be right.

You are looking for two contrasting examples:

- A **convex** profile, one clear, deep minimum. The match is unambiguous.
- A **non-convex** profile, several comparable minima. The match is a coin flip between them.

For reference:

<img src="figures/convex_sample.png" width="350"/>

**Convex profile**

<img src="figures/non_convex_sample.png" width="350"/>

**Non-convex profile**

Your task is to find a region of the Adirondack image that produces each. You will need to pick the
patch location yourself, that search is the exercise, because it forces you to predict from the
image content what the error profile will look like.



In [ ]:
base_path = '../data/adirondack/'
im_left = PIL_resize(load_image(base_path + 'im_left.png'), (0.1, 0.1))
im_right = PIL_resize(load_image(base_path + 'im_right.png'), (0.1, 0.1))

H, W, _ = im_left.shape
print(f'working at {W} x {H} px  (row index 0..{H - 1}, col index 0..{W - 1})')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.imshow(im_left)
ax1.set_title('Left image')
ax2.imshow(im_right)
ax2.set_title('Right image')
plt.tight_layout()
plt.show()


In [ ]:
def error_profile(im_left, im_right, row, col, patch_size, max_search_bound, title):
    """Plot the SAD error between a left patch and every candidate disparity.

    The patch is im_left[row:row+patch_size, col:col+patch_size]. Candidate d is the
    patch at the same rows in the right image shifted d pixels to the LEFT, which is
    exactly the convention calculate_disparity_map uses. The x axis of the plot
    below is disparity.
    """
    assert row is not None and col is not None, 'pick a (row, col) first'
    assert 0 <= row <= im_left.shape[0] - patch_size, f'row must be in [0, {im_left.shape[0] - patch_size}], got {row}'
    assert 0 <= col <= im_left.shape[1] - patch_size, f'col must be in [0, {im_left.shape[1] - patch_size}], got {col}'
    assert max_search_bound > 0, 'max_search_bound must be positive or there is nothing to search'
    
    patch = torch.tensor(im_left[row:row + patch_size, col:col + patch_size, :])

    # d = 0 means "no shift at all", and the search cannot run past the left border.
    d_max = min(max_search_bound, col)
    if d_max < max_search_bound:
        print(f'note: only {d_max} px of search fit to the left of column {col}, '
              f'so the profile stops at d={d_max} instead of {max_search_bound}.')

    errors = np.array([
        sad_similarity_measure(
            patch,
            torch.tensor(im_right[row:row + patch_size, col - d:col - d + patch_size, :])
        ).item()
        for d in range(d_max + 1)
    ])

    search = torch.tensor(im_right[row:row + patch_size, col - d_max:col + patch_size, :])

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 4))
    ax1.imshow(patch)
    ax1.set_title(f'left patch at (row={row}, col={col})')
    ax1.axis('off')
    ax2.imshow(search)
    ax2.set_title('search area in right image')
    ax2.axis('off')
    ax3.plot(errors, '-o', markersize=3)
    ax3.set_title(title)
    ax3.set_xlabel('disparity d (pixels)')
    ax3.set_ylabel('SAD error')
    ax3.grid(True)
    plt.tight_layout()
    plt.show()

    interior_minima = [i for i in range(1, len(errors) - 1)
                       if errors[i] < errors[i - 1] and errors[i] < errors[i + 1]]
    print(f'searched disparities 0..{d_max}; best match at disparity {int(errors.argmin())}')
    print(f'error range [{errors.min():.2f}, {errors.max():.2f}]  '
          f'(ratio {errors.max() / max(errors.min(), 1e-9):.1f}x)')
    print(f'interior local minima: {len(interior_minima)} at {interior_minima}')
    return errors


### Convex error profile

Pick a patch of interest from the left image that will generate a convex error profile.

In [ ]:
# TODO: select a (row, col) for your patch. 
#       adjust max_search_bound if needed.
patch_size = 15
row, col = (50, 148)  # at a globally unique object with unique shapes and colors
max_search_bound = 50

errors_convex = error_profile(
    im_left, im_right,
    row, col,
    patch_size,
    max_search_bound,
    'Convex error profile',
)
# # And of course modifying the figure isn't posible since it wasn't returned and isn't "current" anymore
# plt.gca().set(ylim=[0, None]);

### Reflection Question 5 (3 pts)
Generalize from your example: what kind of image regions produce a convex error profile, and why
does that shape follow from the region's appearance?

**Answer:** A globally (or at least regionally) visually unique object means that matches are unlikely to be found elsewhere. As with Harris corners, detail such as unique colors and distinct shapes helps to make a precise localization and produces a sharp minima.

Note that the error profile is not _truly_ convey, but practically is since local minima have significantly higher disparity at significant distances from the global minima.

I'm not sure how to parse the question "why does that shape follow from the region's appearance?"; I hope I've demonstrated understanding of the topic whether or not I answered the question as it was intended.

### Non-convex error profile

Pick a patch of interest from the left image that will generate a non-convex error profile.

In [ ]:
# TODO: select a (row, col) for your patch. 
#       adjust max_search_bound if needed.
patch_size = 15
row, col = (70, 240)  # in the middle of a repeating pattern
max_search_bound = 50

errors_nonconvex = error_profile(
    im_left, im_right,
    row, col,
    patch_size,
    max_search_bound,
    'Non-convex error profile',
)

### Reflection Question 6 (3 pts)
Generalize: what kind of regions produce a non-convex profile? Name at least two distinct causes,
and say what each does to the curve.

**Answer:**
1. Repeating patterns will have high similarity (low error) at intervals. This shows in the above figure as peaks every 15 pixels - at 15, 30, and 45.
2. Poor detail in the region to match causes proader valleys and more opportunities for false matches. For example, a plain white wall has no repeating pattern but would have high ambiguity and would likely not be convex.

# Part 2: Real stereo images

Now run the full algorithm on real rectified pairs. Each set below highlights a different failure
mode, pay attention to *where* the map goes wrong, not just how good it looks overall.

## Set 1: Adirondack chair

Both similarity measures are computed for each block size so you can compare them directly.

In [ ]:
base_path = '../data/adirondack/'
im_left = PIL_resize(load_image(base_path + 'im_left.png'), (0.1, 0.1))
im_right = PIL_resize(load_image(base_path + 'im_right.png'), (0.1, 0.1))

stereo_helper_fn(
    torch.tensor(im_left),
    torch.tensor(im_right),
    block_size=[5, 9, 13],
    max_search_bound=25
)


### Reflection Question 7 (2 pts)
Which patch size gives the clearest depth representation? Why?  

**Answer:** The "clearest" depth map is the largest patch, which supresses noize quite well. I think the most accurate map is the 5x5 or 9x9 patch, which preserve detail much better (look at the books on the table for the 5x5 patch or the slats on the chair back for the 7x7 patch), at the cost of visual clutter (the chair slats in the 5x5 patch). I think I would choose the 9x9 patch as a good balance of both.

### Reflection Question 8 (3 pts)
In the 5×5 and 9×9 maps, the slats of the chair's backrest come out as a striped patchwork rather
than one smoothly varying surface. Explain what about that part of the scene causes it. 

**Answer:** The plain flat regions seem to be hard to distinguish without any visual features, and so our disparity map is inconclusive and gives erronious results. The 13x13 patch is large enough to always have a bisible feature, and so doesn't have the same trouble.

## Set 2: Bicycle

In [ ]:
base_path = '../data/bicycle/'
im_left = PIL_resize(load_image(base_path + 'im_left.png'), (0.1, 0.1))
im_right = PIL_resize(load_image(base_path + 'im_right.png'), (0.1, 0.1))

stereo_helper_fn(
    torch.tensor(im_left),
    torch.tensor(im_right),
    block_size=[7, 9, 11, 13],          # try 7, 9, 13 as well
    max_search_bound=20
)


## Set 3: Bowling

In [ ]:
base_path = '../data/bowling/'
im_left = PIL_resize(load_image(base_path + 'im_left.png'), (0.2, 0.2))
im_right = PIL_resize(load_image(base_path + 'im_right.png'), (0.2, 0.2))

stereo_helper_fn(
    torch.tensor(im_left),
    torch.tensor(im_right),
    block_size=[9],
    max_search_bound=30
)

In [ ]:
%%needs px
%%depends plotly
from plotly.subplots import make_subplots

disparity = dm.calculate_disparity_map(
    torch.tensor(im_left),
    torch.tensor(im_right),
    9, ssd_similarity_measure, 30,
)

make_subplots(1, 3, shared_xaxes=True, shared_yaxes=True) \
    .add_traces(px.imshow(im_left).data, rows=1, cols=1) \
    .add_traces(px.imshow(im_right).data, rows=1, cols=2) \
    .add_traces(px.imshow(disparity).data, rows=1, cols=3)

### Reflection Question 9 (2 pts)
Look at the tall pin standing just to the right of the ball. Its body comes out at a sensible,
consistent disparity, but its rounded head does not. Compare how the head and the body *look* in
the photograph, and use that to explain why the matcher handles one well and the other badly.

**Answer:** The results look reasonable to me - except a small patch on the left of the neck has an error of about 5 pixels. Comparing the pixel locations between both images and the disparity map in an interactive plot, it looks like a small region of low contrast due to shadows on the pin is slightly more prominent from the right. This causes some ambiguity in the correspondence, but nothing I would consider noteworthy compared to the disparity of the pin body.

## Set 4: Bowling, manipulated

This is the same scene as Set 3, but the images have been altered.

In [ ]:
base_path = '../data/bowling2/'
im_left = PIL_resize(load_image(base_path + 'im_left.png'), (0.2, 0.2))
im_right = PIL_resize(load_image(base_path + 'im_right.png'), (0.2, 0.2))

stereo_helper_fn(
    torch.tensor(im_left),
    torch.tensor(im_right),
    block_size=[9],
    max_search_bound=30
)


### Reflection Question 10 (3 pts)
Compare this map against your Set 3 result. The two scenes are the same, but across the table
surface along the bottom the disparity has collapsed to zero. First say what the manipulation
actually did to the images. Then explain why it collapsed to *zero* rather than to some other
wrong number. 

In [ ]:
%%needs px
px.imshow(im_right.mean(axis=-1) == 1)

**Answer:** The tabletop in both images is saturated (r=g=b=1), and so the similarity function produces the same value for all disparity values. The argmin procedure, using `torch.Tensor.min().indices`, selects the first occurrense of the minimum value which, and so my selected tensor layout of monotonically increasingn dispalrities produces the minimum matching disparity: 0.

## Set 5: Flowers

In [ ]:
base_path = '../data/flowers/'
im_left = PIL_resize(load_image(base_path + 'im_left.png'), (0.10, 0.10))
im_right = PIL_resize(load_image(base_path + 'im_right.png'), (0.10, 0.10))

stereo_helper_fn(
    torch.tensor(im_left),
    torch.tensor(im_right),
    block_size=[9],
    max_search_bound=50
)


**Things to look at** (not graded, these are to guide your eye):

1. The difference in disparity between the flower at the back and its shadow on the wall.
2. The zero-disparity region in the middle of the house structure.
3. How smooth the disparity is across the couch, and what that says about texture.

## Set 6: Stairs



In [ ]:
# Load and keep full resolution (1x scale)
base_path = '../data/stairs/'
im_left = PIL_resize(load_image(base_path + 'im_left.jpg'), (1, 1))
im_right = PIL_resize(load_image(base_path + 'im_right.jpg'), (1, 1))

stereo_helper_fn(
    torch.tensor(im_left),
    torch.tensor(im_right),
    block_size=[3, 5, 7, 11],
    max_search_bound=15
)


### Reflection Question 11 (2 pts)
Why are we able to see a gradual shift in disparity values along the wall? 

**Answer:** The wall must have enough texture for an effective correspondence/similarity search. Since the  wall has a gradual shift in depth, this producess a gradual shift in disparity.

### Reflection Question 12 (2 pts)
What is the effect of block size on your ability to see the stair-like structure in the disparity map?  

**Answer:** The blocck size must be large enough to always include an edge (stair corner), otherwise the  featureless region matches anywhere and defaults to the lowest matched disparity as per [Q10](#Reflection-Question-10-(3-pts)), which is near-zero. Vice versa, if the block is too large, it cannot capture the small detail of the steps' "texture" and the stairs  appear as a smooth surface. rather than the true sawtooth pattern of the steps. 

# Part 3: Smoothing

The maps so far are noisy. Neighbouring pixels on the same physical surface often come out with
very different disparities, because each pixel's disparity was decided **independently**, nothing
in the algorithm knows that surfaces are usually continuous.

Intuitively, adjacent pixels should have similar disparity *unless* there is an object boundary or
an occlusion between them. **Semi-Global Matching (SGM)** builds that intuition into the objective.
Instead of taking the minimum matching cost per pixel, it minimizes a global energy:

\begin{equation*}
E(D) = \sum_{p} \left( C(p, D_p) + \sum_{q} PT(|D_p - D_q|) \right)
\end{equation*}

where $C(p, D_p)$ is the matching cost of assigning disparity $D_p$ to pixel $p$, $q$ ranges over
$p$'s neighbours, and $PT(\cdot)$ penalizes disparity differences between neighbours. The
first term says "match well"; the second says "agree with your neighbours". The result is a
compromise between the two.

Minimizing this exactly over a 2D image is NP-hard, so SGM approximates it by solving the problem
exactly along several 1D paths through each pixel and summing the results, which is where
"semi-global" comes from.

You can read more about how this method works and is optimized here: [Hirschmüller's original paper](https://elib.dlr.de/73119/1/180Hirschmueller.pdf).

### The cost volume

SGM needs more than the winning disparity at each pixel, it needs the cost of *every* candidate,
so it can trade matching quality against smoothness. That is the **cost volume**.

For an input of size (H, W, C) and a maximum disparity of D, the cost volume is a tensor of shape
(H, W, D), where `cost_volume[i, j, d]` is the matching error between the left patch centred at
(i, j) and the right patch shifted by d. It is the same computation as your disparity map, except
you store the whole error profile instead of its argmin.

Implement `calculate_cost_volume()` in `disparity_map.py`. Reuse whatever you can from
`calculate_disparity_map()`. Where the shifted patch would fall outside the image, store a large
default cost (255 is recommended) so those disparities lose.

In [ ]:
from unit_tests.test_disparity_map import test_calculate_cost_volume

print('Cost volume:', verify(test_calculate_cost_volume))


Now run SGM on the Adirondack pair and compare against your unsmoothed result from Set 1.

In [ ]:
from scipy import ndimage

from semiglobalmatching.sgm import sgm

# Same pair, same scale, same block size and the same maximum disparity as Set 1,
# so the smoothing is the only thing that differs between the maps below.
base_path = '../data/adirondack/'
im_left = PIL_resize(load_image(base_path + 'im_left.png'), (0.10, 0.10))
im_right = PIL_resize(load_image(base_path + 'im_right.png'), (0.10, 0.10))

MAX_DISP, BLOCK = 25, 9
pad = BLOCK // 2

# The last three arguments are the maximum disparity, the similarity function,
# and the block size.
sgm_out = np.asarray(sgm(im_left, im_right, 'result', MAX_DISP, sad_similarity_measure, BLOCK))

# sgm() returns a full-size map, while the block matcher drops a border of
# BLOCK//2. Crop so all three maps below are the same size, pixel for pixel.
sgm_raw = sgm_out[pad:-pad, pad:-pad]


unsmoothed = calculate_disparity_map(torch.tensor(im_left), torch.tensor(im_right),
                                     BLOCK, sad_similarity_measure, MAX_DISP).numpy()

fig, axes = plt.subplots(1, 2, figsize=(17, 5))
panels = [(unsmoothed, 'No smoothing (as in Set 1)'),
          (sgm_raw, 'SGM only')]
for ax, (d, t) in zip(axes, panels):
    img = ax.imshow(d, cmap='jet', interpolation='nearest', vmin=0, vmax=MAX_DISP)
    ax.set_title(t)
    ax.set_axis_off()
    fig.colorbar(img, ax=ax, shrink=0.7)
plt.tight_layout()
plt.show()

# A crude numerical handle on "how smooth is this map": the average jump in
# disparity between horizontally adjacent pixels.
for d, t in panels:
    print(f'{t:<30} mean |neighbour difference| = {np.abs(np.diff(d, axis=1)).mean():.3f}')


### Reflection Question 13 (3 pts)
The maps above use the same pair, scale, block size and maximum disparity, so smoothing is
the only difference between them. Describe what specifically changed between the unsmoothed map
and the SGM one, and explain the mechanism.

**Answer:** *your answer here*

### Reflection Question 14 (2 pts)
Which regions does smoothing improve, and why does the smoothness assumption hold there?

**Answer:** *your answer here*

### Reflection Question 15 (2 pts)
Which regions does smoothing make worse, and why does the smoothness assumption fail there?

**Answer:** *your answer here*

### Reflection Question 16 (2 pts)
Would this approach still work if the two images differed by both a horizontal **and** a vertical
shift? Explain in terms of what the algorithm assumes about where the match can be.

**Answer:** *your answer here*

# Part 4: Extra credit — how much should you trust a disparity map? (+10 pts)

Something has come up several times now. In Set 4 the disparity collapsed to zero and the map
said nothing about it. Along the left border of every map there is a band of values that are
completely made up. The chair back comes out striped, and the map reports those stripes with the
same confidence as everything else. A disparity map is just a grid of numbers, and nothing in it
says which numbers to believe.

But the information is there, and you already computed it. `calculate_disparity_map()` finds the
best disparity and throws the rest of the error profile away. The cost volume keeps all of it, and
the shape of a pixel's profile tells you how much its answer is worth. One deep minimum on its own
means the match was clear. A flat or bumpy profile means the matcher was guessing.

You will write two ways of scoring that, and then check whether the scores mean anything: throw
away the pixels that score badly and see whether the ones left over are any better.

### What to implement (4 pts)

`confidence_from_cost_volume()`  in `disparity_map.py`, with both methods described in its
docstring. The unit tests are in `test_disparity_map.py`. Read the docstring carefully.

In [ ]:
# Same pair and settings as Set 1 and Part 3, so everything here lines up with the
# maps you have already looked at.
base_path = '../data/adirondack/'
im_left = PIL_resize(load_image(base_path + 'im_left.png'), (0.10, 0.10))
im_right = PIL_resize(load_image(base_path + 'im_right.png'), (0.10, 0.10))

MAX_DISP, BLOCK = 25, 9
pad = BLOCK // 2

# calculate_cost_volume takes the NUMBER of disparities, so 26 covers d = 0 .. 25.
cost_volume = calculate_cost_volume(torch.tensor(im_left), torch.tensor(im_right),
                                    MAX_DISP + 1, sad_similarity_measure, BLOCK)
print('cost volume shape:', tuple(cost_volume.shape))


### Run the confidence tests

In [ ]:
from unit_tests.test_disparity_map import (test_confidence_peak_ratio,
                                           test_confidence_curvature)

print('Peak ratio:', verify(test_confidence_peak_ratio))
print('Curvature: ', verify(test_confidence_curvature))


### Look at the confidence maps

The two methods are on different scales, so do not compare their numbers against each other
directly. Compare each map against itself: which parts of this scene does it call reliable, and
which does it flag?

In [ ]:
conf_peak = confidence_from_cost_volume(cost_volume, method='peak_ratio').numpy()
conf_curv = confidence_from_cost_volume(cost_volume, method='curvature').numpy()

# Trim the block border so these line up with the disparity maps from Set 1.
disparity = cost_volume.argmin(dim=2).numpy()[pad:-pad, pad:-pad]
conf_peak = conf_peak[pad:-pad, pad:-pad]
conf_curv = conf_curv[pad:-pad, pad:-pad]

# Both measures have a long upper tail, so the colour scale is cut at the 95th
# percentile; otherwise a handful of pixels wash the rest of the map out.
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
panels = [(disparity, 'Disparity', 'jet', None),
          (conf_peak, 'Confidence: peak ratio', 'viridis', 95),
          (conf_curv, 'Confidence: curvature', 'viridis', 95)]
for ax, (data, title, cmap, clip) in zip(axes, panels):
    img = ax.imshow(data, cmap=cmap,
                    vmax=np.percentile(data, clip) if clip else None)
    ax.set_title(title)
    ax.set_axis_off()
    fig.colorbar(img, ax=ax, shrink=0.7)
plt.tight_layout()
plt.show()

for name, c in (('peak ratio', conf_peak), ('curvature', conf_curv)):
    print(f'{name:<12} median {np.median(c):6.2f}   10th pct {np.percentile(c, 10):6.2f}'
          f'   90th pct {np.percentile(c, 90):6.2f}')

# Average confidence over four regions, for Question 17. The slices are (rows, cols)
# into the maps above.
regions = {'chair back slats': (slice(6, 66), slice(211, 271)),
           'books and box':    (slice(66, 106), slice(56, 116)),
           'draped cloth':     (slice(140, 180), slice(26, 76)),
           'left border':      (slice(None), slice(0, 11))}
print()
print(f'{"region":<20}{"peak ratio":>12}{"curvature":>12}')
for name, sl in regions.items():
    print(f'{name:<20}{conf_peak[sl].mean():>12.2f}{conf_curv[sl].mean():>12.2f}')


### Reflection Question 17 (3 pts)
Using the maps and the per-region numbers printed above:

1. Pick one region each measure handles sensibly, and say why the score came out the way it did.
2. The two measures disagree about the chair back. One of them rates it among the most reliable
   parts of the image and the other does not. Say which is which, which one you believe, and what
   it is in each formula that explains the disagreement. 

**Answer:** *your answer here*

### Does the score mean anything?

A confidence map is only useful if the pixels it likes really are better than the pixels it does
not. The cell below checks that. It sorts the pixels by score, keeps the top fraction, and reports
how many of the ones left over agree with the SGM map from Part 3.

Keeping fewer pixels should push the agreement up. The catch is that you would also expect *some*
improvement from keeping fewer pixels at random, so a random score is included as a control.

In [ ]:
# There is no ground truth for this scene, so we use the SGM map from Part 3 as a
# stand-in reference. It is not perfect, but it is a lot better than the raw block
# matcher, so "agrees with SGM" is a reasonable stand-in for "probably right".
try:
    sgm_raw
except NameError:
    raise RuntimeError('Run the SGM cell in Part 3 first -- this section reuses its result.')

reference = sgm_raw[:disparity.shape[0], :disparity.shape[1]]
agrees = np.abs(disparity - reference) <= 2

# The random score is the control. A confidence measure that cannot beat shuffling
# the pixels is not telling us anything, and without this line on the plot there is
# no way to tell the difference.
rng = np.random.default_rng(0)
scores = {'peak ratio': conf_peak,
          'curvature': conf_curv,
          'random (control)': rng.random(disparity.shape)}

fractions = [1.0, 0.9, 0.75, 0.5, 0.25, 0.1]
curves = {name: [] for name in scores}

print(f'{"kept":>6}' + ''.join(f'{name:>20}' for name in scores))
for frac in fractions:
    row = []
    for name, score in scores.items():
        keep = score >= np.quantile(score, 1 - frac)
        pct = agrees[keep].mean() * 100
        curves[name].append(pct)
        row.append(pct)
    print(f'{frac * 100:>5.0f}%' + ''.join(f'{v:>19.1f}%' for v in row))

plt.figure(figsize=(7, 5))
for name, ys in curves.items():
    plt.plot([f * 100 for f in fractions], ys, '-o', label=name)
plt.xlabel('percentage of pixels kept')
plt.ylabel('percentage agreeing with the SGM reference')
plt.title('What you gain by dropping the least confident pixels')
plt.gca().invert_xaxis()
plt.grid(True)
plt.legend()
plt.show()


### Reflection Question 18 (3 pts)
1. Describe the shape of the three curves. What does the flat random line tell you, and what would
   it have meant if all three lines had looked the same?
2. The two real measures do not perform identically. Which one would you use, and does the ranking
   match what you said in Q17?

**Answer:** *your answer here*

# Part 5: AI-Use Disclosure (required)

See the syllabus for the details of the course AI policy.

Answer the questions below. If you didn't use any AI tools, write "None" for the first question and
skip the second.

### Your disclosure

**1. Which AI tools did you use, if any?**
*(Name the model, the version, the subscription plan, code editor, and any related AI tools. Write "None" if you used none.)*


**2. What did you use them for, and which parts of this notebook did they influence?**
*Be specific about both.
List the cells, functions, or written answers involved.*

---

**Statement.** *By submitting this notebook you confirm that the code and written answers in it are your
own work, that you did not use AI tools to solve any part of this assignment, and that any other AI
use is fully described above.*


---

**Before you submit:** restart the kernel, run every cell from top to bottom,
confirm that every figure and printed result is visible, save the notebook, then run
`python zip_submission.py --uid <your_uid>` from the repo root.